# London's Reservoirs: 37 Years of Water, One Day at a Time

A curious dig through daily reservoir levels for London's two big storage groups, the **Lower Lee** and the **Lower Thames**, from **1 January 1989** to **31 May 2026**.

Each number is *percent of capacity*: how full the reservoirs were that day. That's it. One tidy column per group, one row per day, ~13,665 days. The question I keep coming back to: **what does 37 years of a city's water look like when you plot every single day?**

---

### Data & licence

- **London Datastore, London Reservoir Levels**: https://data.london.gov.uk/dataset/london-reservoir-levels-24ry5
- **Source: Environment Agency / Defra, Water situation reports for England**: https://www.gov.uk/government/collections/water-situation-reports-for-england
- Contains public sector information licensed under the **Open Government Licence v2.0**: https://www.nationalarchives.gov.uk/doc/open-government-licence/version/2/

> The raw CSV lives under `../data/` and is gitignored; grab it yourself from the London Datastore link above.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)

GROUPS = {"lower_lee_group": "Lower Lee", "lower_thames_group": "Lower Thames"}
COLORS = {"Lower Lee": "#2a9d8f", "Lower Thames": "#e76f51"}

## 1. Load and clean

A few little gotchas hide in this file, and finding them is half the fun:

1. **Column headers have stray spaces**: `" lower_lee_group "` rather than `"lower_lee_group"`.
2. **Dates are in two formats**: most rows read like `01-Jan-89`, but a stretch in the middle switched to `01/06/2020`. `format="mixed", dayfirst=True` handles both.
3. **A handful of non-numeric cells**: `n/a` and `---` where a reading was missing. 16 out of ~27,000 values. We coerce those to `NaN` and keep going.

In [2]:
raw = pd.read_csv("../data/london_reservoir_levels.csv")
raw.columns = raw.columns.str.strip()  # gotcha #1

df = raw[["date", *GROUPS]].copy()
df["date"] = pd.to_datetime(df["date"], format="mixed", dayfirst=True)  # gotcha #2

for col in GROUPS:  # gotcha #3
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)
df.head()

,date,lower_lee_group,lower_thames_group
0,1989-01-01,83.0,76.0
1,1989-01-02,83.0,75.0
2,1989-01-03,83.0,75.0
3,1989-01-04,82.0,75.0
4,1989-01-05,82.0,75.0


In [3]:
n_days = (df["date"].max() - df["date"].min()).days + 1
print(f"Rows                : {len(df):,}")
print(f"Date range          : {df['date'].min():%Y-%m-%d} -> {df['date'].max():%Y-%m-%d}")
print(f"Calendar days spanned: {n_days:,}")
print(f"Duplicate dates     : {df['date'].duplicated().sum()}")
print(f"Missing daily reads : {n_days - len(df)}")
print()
print("Missing (NaN) values after cleaning:")
print(df[list(GROUPS)].isna().sum())
print()
df[list(GROUPS)].describe().round(1)

Rows                : 13,665
Date range          : 1989-01-01 -> 2026-05-31
Calendar days spanned: 13,665
Duplicate dates     : 0
Missing daily reads : 0

Missing (NaN) values after cleaning:
lower_lee_group        3
lower_thames_group    13
dtype: int64



,lower_lee_group,lower_thames_group
count,13662.0,13652.0
mean,87.2,88.1
std,10.4,11.6
min,48.0,42.0
25%,82.0,84.0
50%,90.0,92.0
75%,95.0,96.0
max,100.0,100.0


A complete daily record: one reading per day for 37 years, no gaps in the calendar. That's rare and lovely. The two series don't behave the same, even in the summary stats: the Lower Thames sits fuller on average and seldom drops as low as the Lee.

## 2. The whole thing, every day

Before slicing anything, just plot all 13,665 days. Drag to zoom, double-click to reset.

In [4]:
long = df.melt("date", value_vars=list(GROUPS), var_name="group", value_name="pct")
long["group"] = long["group"].map(GROUPS)

fig = px.line(
    long, x="date", y="pct", color="group",
    color_discrete_map=COLORS,
    labels={"pct": "% of capacity", "date": "", "group": ""},
    title="London reservoir levels, 1989–2026 (daily % of capacity)",
)
fig.add_hline(y=100, line_dash="dot", line_color="#999", annotation_text="full")
fig.update_layout(template="plotly_white", hovermode="x unified", legend_title="")
fig_daily = fig
fig.show()

The sawtooth is the story: **fill through winter, draw down through summer, repeat**. Some summers bite deeper than others. The obvious dips (the mid-1990s, 2005–06, and 2022) are the droughts you might remember from the news.

## 3. The heartbeat: an average year

If every year follows the same fill-and-draw rhythm, we should be able to fold all 37 years onto a single 12-month clock and see the pulse. This is the *climatology*: the typical shape of a year.

In [9]:
df["month"] = df["date"].dt.month
clim = df.groupby("month")[list(GROUPS)].agg(["mean", "min", "max"])
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
BAND = {"Lower Lee": "rgba(42,157,143,0.15)", "Lower Thames": "rgba(231,111,81,0.15)"}

fig = go.Figure()
for col, name in GROUPS.items():
    fig.add_trace(go.Scatter(x=month_names, y=clim[(col, "max")], line=dict(width=0),
                             showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=month_names, y=clim[(col, "min")], fill="tonexty", line=dict(width=0),
                             fillcolor=BAND[name], name=f"{name} range", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=month_names, y=clim[(col, "mean")], name=name,
                             line=dict(color=COLORS[name], width=3), mode="lines+markers"))

fig.update_layout(template="plotly_white", title="The average London water year (mean % by month, 1989–2026)",
                  yaxis_title="% of capacity", hovermode="x unified")
fig_clim = fig
fig.show()

There's the heartbeat: reservoirs peak in **late winter / early spring** and bottom out in **autumn** (around September–October) before the winter rains refill them. The Lower Thames rides higher than the Lower Lee year in, year out. That gap is structural, built into the two systems.

## 4. Which years ran driest?

The single most useful number for a drought is the **annual minimum**: how low did it get at the worst point of that year? Lower bars = tighter years.

In [ ]:
df["year"] = df["date"].dt.year
annual_min = df.groupby("year")[list(GROUPS)].min().rename(columns=GROUPS)
# 2026 is a partial year (to May), flag it but keep it
annual_min_long = annual_min.reset_index().melt("year", var_name="group", value_name="min_pct")

fig = px.bar(annual_min_long, x="year", y="min_pct", color="group", barmode="group",
             color_discrete_map=COLORS, labels={"min_pct": "lowest % reached", "year": "", "group": ""},
             title="Lowest level reached each year (annual minimum)")
fig.update_layout(template="plotly_white", hovermode="x unified")
fig_annual_min = fig
fig.show()

print("5 driest years by annual minimum:")
for name in GROUPS.values():
    print(f"\n{name}:")
    print(annual_min[name].nsmallest(5).to_string())

5 driest years by annual minimum:

Lower Lee:
year
1991    48.0
1992    50.0
2003    51.0
2011    55.0
1999    58.0

Lower Thames:
year
1996    42.0
1997    44.0
2003    46.0
1990    48.0
2022    50.0


## 5. Zoom on a drought: 2022

Here's the twist the annual-minimum chart just handed us. **2022** is the drought everyone remembers: the hosepipe bans, the brown lawns, the official drought declaration across England. But by the numbers above, the Lower Thames only reached **50%** that year, its *fifth*-lowest on record. The brutal years were **1996 (42%)** and **1997 (44%)**, a slower, deeper squeeze that faded from public memory.

So let's put 2022 under the microscope against the 37-year *normal band* (the min–max envelope of every other year) and ask the honest question: how unusual was it?

In [7]:
focus_year = 2022
df["doy"] = df["date"].dt.dayofyear
col = "lower_thames_group"
name = GROUPS[col]

hist = df[df["year"] != focus_year].groupby("doy")[col].agg(["min", "max", "mean"])
this = df[df["year"] == focus_year].set_index("doy")[col]

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist.index, y=hist["max"], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=hist.index, y=hist["min"], fill="tonexty", fillcolor="rgba(150,150,150,0.25)",
                         line=dict(width=0), name="1989–2026 range"))
fig.add_trace(go.Scatter(x=hist.index, y=hist["mean"], line=dict(color="#666", dash="dash"), name="typical year"))
fig.add_trace(go.Scatter(x=this.index, y=this.values, line=dict(color=COLORS[name], width=3), name=f"{focus_year}"))
fig.update_layout(template="plotly_white", title=f"{name}: {focus_year} vs. the 37-year normal band",
                  xaxis_title="day of year", yaxis_title="% of capacity", hovermode="x unified")
fig_band = fig
fig.show()

## 6. Where these two groups sit

The dataset never says *where* these reservoirs are: "Lower Lee" and "Lower Thames" are just two column names. They're real bodies of water, and they sit in two very different corners of London. The **Lower Lee** group is the Lee Valley chain up in the north-east (Tottenham, Walthamstow, Chingford); the **Lower Thames** group is the cluster of big reservoirs out west, towards Staines and Heathrow.

To draw them, I pulled the reservoir outlines straight from **OpenStreetMap** via the Overpass API and coloured each polygon by its group. No coordinates live in the original CSV, so this is a separate open-data layer stitched on top.

> Reservoir geometry © OpenStreetMap contributors, licensed under the [Open Database Licence (ODbL)](https://www.openstreetmap.org/copyright). Basemap tiles © OpenStreetMap contributors.

In [15]:
# Reservoir outlines from OpenStreetMap via the Overpass API.
# Data: © OpenStreetMap contributors, licensed under the Open Database Licence (ODbL).
# https://www.openstreetmap.org/copyright
import json
import math
import urllib.parse
import urllib.request

# Each group as a bounding box (South, West, North, East)
CLUSTERS = {
    "Lower Lee":    (51.565, -0.075, 51.665,  0.010),
    "Lower Thames": (51.350, -0.590, 51.490, -0.340),
}
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
MIN_AREA_M2 = 80_000            # drop ornamental ponds and filter beds
EXCLUDE = ("Balancing",)        # airport/stormwater drainage, not water supply


def fetch_water_bodies(bbox):
    s, w, n, e = bbox
    query = f"""
    [out:json][timeout:120];
    (
      way["water"="reservoir"]({s},{w},{n},{e});
      way["landuse"="reservoir"]({s},{w},{n},{e});
      way["natural"="water"]["name"~"Reservoir"]({s},{w},{n},{e});
      relation["water"="reservoir"]({s},{w},{n},{e});
      relation["natural"="water"]["name"~"Reservoir"]({s},{w},{n},{e});
    );
    out geom;
    """
    payload = urllib.parse.urlencode({"data": query}).encode()
    req = urllib.request.Request(
        OVERPASS_URL, data=payload,
        headers={"User-Agent": "london-reservoir-notebook/1.0 (open data viz)"},
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.load(resp)


def _stitch(ways):
    """Join member ways (shared endpoints) into a single ordered ring."""
    ways = [list(w) for w in ways if w]
    if not ways:
        return []
    ring = ways.pop(0)
    while ways:
        for i, w in enumerate(ways):
            if w[0] == ring[-1]:
                ring += w[1:]; ways.pop(i); break
            if w[-1] == ring[-1]:
                ring += list(reversed(w))[1:]; ways.pop(i); break
        else:
            ring += ways.pop(0)  # disjoint piece; append as-is
    return ring


def rings_from_element(el):
    """Return outer rings as lists of (lon, lat). Relations are stitched."""
    if el["type"] == "way" and el.get("geometry"):
        return [[(p["lon"], p["lat"]) for p in el["geometry"]]]
    outer = [[(p["lon"], p["lat"]) for p in m["geometry"]]
             for m in el.get("members", [])
             if m.get("role") == "outer" and m.get("geometry")]
    stitched = _stitch(outer)
    return [stitched] if stitched else []


def ring_area_m2(ring):
    """Rough planar area via local equirectangular projection."""
    if len(ring) < 4:
        return 0.0
    lat0 = math.radians(sum(p[1] for p in ring) / len(ring))
    mx, my = 111_320 * math.cos(lat0), 110_540
    xy = [(lon * mx, lat * my) for lon, lat in ring]
    a = sum(xy[i][0] * xy[i + 1][1] - xy[i + 1][0] * xy[i][1]
            for i in range(len(xy) - 1))
    return abs(a) / 2.0


reservoirs, seen = [], set()
for group, bbox in CLUSTERS.items():
    for el in fetch_water_bodies(bbox)["elements"]:
        name = el.get("tags", {}).get("name", "(unnamed)")
        if any(x in name for x in EXCLUDE):
            continue
        for ring in rings_from_element(el):
            area = ring_area_m2(ring)
            if area < MIN_AREA_M2 and not name.endswith("Reservoir"):
                continue
            clon = sum(p[0] for p in ring) / len(ring)
            clat = sum(p[1] for p in ring) / len(ring)
            key = (round(clon, 4), round(clat, 4))
            if key in seen:
                continue
            seen.add(key)
            reservoirs.append({"name": name, "group": group, "ring": ring,
                               "lon": clon, "lat": clat, "area_km2": area / 1e6})

print(f"Kept {len(reservoirs)} reservoir polygons from OpenStreetMap")
for group in CLUSTERS:
    grp = [r for r in reservoirs if r["group"] == group]
    named = sorted({r["name"] for r in grp if r["name"] != "(unnamed)"})
    print(f"\n{group} ({len(grp)} polygons):")
    print("  " + ", ".join(named))


Kept 26 reservoir polygons from OpenStreetMap

Lower Lee (10 polygons):
  Banbury Reservoir, High Maynard Reservoir, King George V Reservoir, Lockwood Reservoir, Low Maynard Reservoir, Reservoir №4, Reservoir №5, Warwick Reservoir East, Warwick Reservoir West, William Girling Reservoir

Lower Thames (16 polygons):
  Bessborough Reservoir, Carp Trench Reservoir, Grand Junction Reservoir, Island Barn Reservoir, King George VI Reservoir, Knight Reservoir, Queen Elizabeth II Reservoir, Queen Mary Reservoir, Red House Reservoir, Stain Hill East Reservoir, Stain Hill West Reservoir, Staines Reservoirs, Sunnyside Reservoir, The Queen Mother Reservoir, Wraysbury Reservoir


In [16]:
MAP_FILL = {"Lower Lee": "rgba(42,157,143,0.45)", "Lower Thames": "rgba(231,111,81,0.45)"}

fig = go.Figure()
for group in CLUSTERS:
    grp = [r for r in reservoirs if r["group"] == group]

    # filled polygon outlines (rings separated by None)
    lons, lats = [], []
    for r in grp:
        lons.extend([p[0] for p in r["ring"]] + [None])
        lats.extend([p[1] for p in r["ring"]] + [None])
    fig.add_trace(go.Scattermap(
        lon=lons, lat=lats, mode="lines", fill="toself",
        fillcolor=MAP_FILL[group], line=dict(color=COLORS[group], width=1.5),
        name=group, hoverinfo="skip", legendgroup=group,
    ))

    # one hover dot per reservoir, carrying its name
    fig.add_trace(go.Scattermap(
        lon=[r["lon"] for r in grp], lat=[r["lat"] for r in grp],
        mode="markers", marker=dict(size=7, color=COLORS[group]),
        text=[r["name"] for r in grp], hovertemplate="%{text}<extra>" + group + "</extra>",
        name=group, legendgroup=group, showlegend=False,
    ))

fig.update_layout(
    map=dict(style="open-street-map", center=dict(lat=51.51, lon=-0.29), zoom=8.9),
    title="Where London's water sits: Lower Lee vs. Lower Thames reservoir groups",
    legend=dict(title="", orientation="h", yanchor="bottom", y=0.01, xanchor="left", x=0.01,
                bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(l=0, r=0, t=50, b=0),
)
fig_map = fig
fig.show()


## 7. Takeaways

A few things that jumped out from 37 years of daily readings:

- **The record is complete, start to finish**: 37 years, one reading a day, zero calendar gaps. Only 16 missing values total.
- **Water has a heartbeat**: fill in winter, draw down to an autumn low, every single year.
- **The Lower Thames runs fuller than the Lower Lee**: a structural gap that holds year after year.
- **The drought you remember isn't the worst one.** 2022 made headlines, but 1996–97 (and 1991 for the Lee) drew the reservoirs down further. Memory and data disagree, and the data wins.
- **The normal-band view** is the most honest way to show a drought: it answers "how unusual was this?"

The single most surprising finding: *the summer everyone remembers as the big drought wasn't London's driest.*

## 8. Save the interactive charts

Export the figures that carry the story (the full daily series, the average year, the annual-minimum bars, the 2022 normal-band view, and the reservoir map) as standalone interactive HTML (Plotly loaded from CDN). Handy for embedding anywhere a browser can reach.

In [ ]:
from pathlib import Path

EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(exist_ok=True)

VIEWPORT_META = '<meta name="viewport" content="width=device-width, initial-scale=1" />'


def inject_viewport(path: Path) -> None:
    """Plotly's write_html omits a viewport meta; add one so embeds render
    at device width inside iframes on phones/tablets."""
    html = path.read_text()
    if 'name="viewport"' in html:
        return
    html = html.replace('<meta charset="utf-8" />', '<meta charset="utf-8" />' + VIEWPORT_META, 1)
    path.write_text(html)


html_config = {"responsive": True, "displaylogo": False}
charts = {
    "daily-levels": fig_daily,
    "average-year": fig_clim,
    "annual-minimum": fig_annual_min,
    "drought-2022-band": fig_band,
    "reservoir-map": fig_map,
}
for slug, figure in charts.items():
    if slug == "reservoir-map":
        figure.update_layout(autosize=True, margin=dict(l=0, r=0, t=50, b=0))
    else:
        figure.update_layout(autosize=True, margin=dict(l=60, r=20, t=60, b=40))
    out = EXPORT_DIR / f"{slug}.html"
    figure.write_html(out, include_plotlyjs="cdn", full_html=True, config=html_config)
    inject_viewport(out)
    print(f"wrote {out}")


wrote exports/daily-levels.html
wrote exports/average-year.html
wrote exports/annual-minimum.html
wrote exports/drought-2022-band.html
wrote exports/reservoir-map.html
